# Phase II – Step 2: Base models on UNSW-NB15 (Mouwiya official split)
Loads the saved `train.csv` / `test.csv`, runs data checks, carves a validation set (for later calibration), and trains RF (paper config), a stronger RF, gradient boosting and logistic regression. Metrics are reported on validation and test, plus per-attack-category recall.

In [13]:
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
pd.set_option('display.width', 200)

# OUT_DIR = REPO_ROOT / 'data' / 'phaseII_outputs'
# OUT_DIR.mkdir(parents=True, exist_ok=True)
# print('data dir:', DATA_DIR, '| exists:', DATA_DIR.is_dir())# Static paths (WSL Ubuntu view of the D: drive)
DATA_DIR = Path('/mnt/d/ai_security/xai-q-audit-ids/data/unsw-nb15_mouwiya')
OUT_DIR = Path('/mnt/d/ai_security/xai-q-audit-ids/data/phaseII_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('data dir:', DATA_DIR, '| exists:', DATA_DIR.is_dir())

data dir: /mnt/d/ai_security/xai-q-audit-ids/data/unsw-nb15_mouwiya | exists: True


## 1. Load saved data and sanity checks

In [14]:

train_raw = pd.read_csv(DATA_DIR / 'train.csv')
test_raw = pd.read_csv(DATA_DIR / 'test.csv')
print('train', train_raw.shape, '| test', test_raw.shape)
print('same columns:', list(train_raw.columns) == list(test_raw.columns))
print('missing values: train', int(train_raw.isna().sum().sum()), '| test', int(test_raw.isna().sum().sum()))

FEATURE_SRC = [c for c in train_raw.columns if c not in ('id', 'attack_cat', 'label')]
dup_train = train_raw.duplicated(FEATURE_SRC).sum()
dup_test = test_raw.duplicated(FEATURE_SRC).sum()
overlap = pd.merge(train_raw[FEATURE_SRC].drop_duplicates(), test_raw[FEATURE_SRC].drop_duplicates(), on=FEATURE_SRC).shape[0]
print(f'duplicate feature rows: train {dup_train:,} | test {dup_test:,}')
print(f'identical feature vectors shared by train and test: {overlap:,}')
print('\nlabel share (attack=1): train %.3f | test %.3f' % (train_raw.label.mean(), test_raw.label.mean()))
pd.concat([train_raw.attack_cat.value_counts().rename('train'), test_raw.attack_cat.value_counts().rename('test')], axis=1)


train (175341, 45) | test (82332, 45)
same columns: True
missing values: train 0 | test 0
duplicate feature rows: train 74,301 | test 28,386
identical feature vectors shared by train and test: 1,302

label share (attack=1): train 0.681 | test 0.551


,train,test
attack_cat,,
Normal,56000,37000
Generic,40000,18871
Exploits,33393,11132
Fuzzers,18184,6062
DoS,12264,4089
Reconnaissance,10491,3496
Analysis,2000,677
Backdoor,1746,583
Shellcode,1133,378


## 2. Preprocessing
- Drop `id` and the raw TCP sequence numbers `stcpb`/`dtcpb` (random per connection, no behavioural meaning; set `DROP_SEQ=False` to keep them).
- One-hot `proto`, `service`, `state` with categories fixed from train; unseen test values become all-zeros.
- Drop duplicate rows in train only (test is left as provided).
- Validation set = 20% stratified (by `attack_cat`) of train; it is never used for fitting and is reserved for calibration/thresholds in later steps.

In [15]:
DROP_SEQ = True
CATEGORICAL = ['proto', 'service', 'state']
drop_cols = ['id', 'attack_cat', 'label'] + (['stcpb', 'dtcpb'] if DROP_SEQ else [])

train_df = train_raw.drop_duplicates(FEATURE_SRC).reset_index(drop=True)
print('train rows after dedupe:', len(train_df), '(removed', len(train_raw) - len(train_df), ')')

fit_df, val_df = train_test_split(train_df, test_size=0.2, stratify=train_df['attack_cat'], random_state=SEED)
categories = {c: sorted(fit_df[c].astype(str).unique()) for c in CATEGORICAL}


def encode(df):
    X = df.drop(columns=drop_cols).copy()
    dummies = [pd.DataFrame({f'{c}={v}': (X[c].astype(str) == v).astype(np.int8) for v in cats}, index=X.index)
               for c, cats in categories.items()]
    X = X.drop(columns=CATEGORICAL)
    return pd.concat([X, *dummies], axis=1).astype(np.float32)


X_fit, X_val, X_test = encode(fit_df), encode(val_df), encode(test_raw)
y_fit, y_val, y_test = fit_df['label'].values, val_df['label'].values, test_raw['label'].values
unseen = {c: sorted(set(test_raw[c].astype(str)) - set(categories[c])) for c in CATEGORICAL}
print('features:', X_fit.shape[1], '| fit', X_fit.shape, '| val', X_val.shape, '| test', X_test.shape)
print('categories unseen in train:', {k: v for k, v in unseen.items() if v} or 'none')

train rows after dedupe: 101040 (removed 74301 )
features: 190 | fit (80832, 190) | val (20208, 190) | test (82332, 190)
categories unseen in train: {'state': ['ACC', 'CLO']}


## 3. Train models

In [16]:
models = {
    'RF-paper (50 trees, depth 10)': RandomForestClassifier(n_estimators=50, max_depth=10, n_jobs=-1, random_state=SEED),
    'RF-large (300 trees)': RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=SEED),
    'HistGradientBoosting': HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, random_state=SEED),
    'LogisticRegression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED)),
}
fit_time = {}
for name, m in models.items():
    t = time.time()
    m.fit(X_fit, y_fit)
    fit_time[name] = time.time() - t
    print(f'{name:<32s} fitted in {fit_time[name]:.1f}s')

RF-paper (50 trees, depth 10)    fitted in 0.9s
RF-large (300 trees)             fitted in 14.7s
HistGradientBoosting             fitted in 14.6s
LogisticRegression               fitted in 6.9s


## 4. Evaluate (validation and test)

In [17]:
def metrics(y, p, s):
    return dict(acc=accuracy_score(y, p), precision=precision_score(y, p), recall=recall_score(y, p),
                f1=f1_score(y, p), auc=roc_auc_score(y, s))


rows, scores = [], {}
for name, m in models.items():
    for split, X, y in (('val', X_val, y_val), ('test', X_test, y_test)):
        s = m.predict_proba(X)[:, 1]
        scores[(name, split)] = s
        rows.append(dict(model=name, split=split, **metrics(y, (s >= 0.5).astype(int), s)))
results = pd.DataFrame(rows).round(4)
# results.to_csv(OUT_DIR / 'base_model_metrics.csv', index=False)
results

,model,split,acc,precision,recall,f1,auc
0,"RF-paper (50 trees, depth 10)",val,0.9070,0.8453,0.9905,0.9121,0.9796
1,"RF-paper (50 trees, depth 10)",test,0.8211,0.7554,0.9983,0.8600,0.9775
2,RF-large (300 trees),val,0.9317,0.9048,0.9609,0.9320,0.9853
3,RF-large (300 trees),test,0.8649,0.8083,0.9890,0.8896,0.9822
4,HistGradientBoosting,val,0.9334,0.9158,0.9508,0.9330,0.9863
5,HistGradientBoosting,test,0.8723,0.8204,0.9835,0.8946,0.9842
6,LogisticRegression,val,0.8947,0.8359,0.9755,0.9003,0.9660
7,LogisticRegression,test,0.8091,0.7518,0.9753,0.8491,0.9555


In [18]:
# Confusion matrices on test (rows = actual, cols = predicted; 0 = normal, 1 = attack)
for name in models:
    cm = confusion_matrix(y_test, (scores[(name, 'test')] >= 0.5).astype(int))
    print(f'{name}\n{cm}\n')

RF-paper (50 trees, depth 10)
[[22350 14650]
 [   78 45254]]

RF-large (300 trees)
[[26370 10630]
 [  497 44835]]

HistGradientBoosting
[[27237  9763]
 [  748 44584]]

LogisticRegression
[[22406 14594]
 [ 1120 44212]]



In [19]:
# Per-attack-category detection rate on test (share of each category predicted as attack)
cats = test_raw['attack_cat'].values
per_cat = {}
for name in models:
    pred = (scores[(name, 'test')] >= 0.5).astype(int)
    per_cat[name] = {c: (pred[cats == c] == (0 if c == 'Normal' else 1)).mean() for c in sorted(set(cats))}
per_cat = pd.DataFrame(per_cat).round(3)
per_cat.insert(0, 'n_test', pd.Series(cats).value_counts())
# per_cat.to_csv(OUT_DIR / 'per_category_detection_test.csv')
per_cat

,n_test,"RF-paper (50 trees, depth 10)",RF-large (300 trees),HistGradientBoosting,LogisticRegression
Analysis,677,1.000,1.000,0.975,0.914
Backdoor,583,1.000,1.000,1.000,0.988
DoS,4089,1.000,0.999,0.995,0.979
Exploits,11132,1.000,0.997,0.995,0.950
Fuzzers,6062,0.988,0.925,0.893,0.951
Generic,18871,1.000,1.000,1.000,0.995
Normal,37000,0.604,0.713,0.736,0.606
Reconnaissance,3496,1.000,0.999,0.999,0.996
Shellcode,378,1.000,0.995,0.992,0.987
Worms,44,1.000,1.000,1.000,1.000


## 5. Save models and artefacts for the next steps

In [ ]:
joblib.dump(models, OUT_DIR / 'base_models.joblib')
joblib.dump(dict(feature_columns=list(X_fit.columns), categories=categories, drop_cols=drop_cols), OUT_DIR / 'preprocessing.joblib')
X_fit.assign(label=y_fit, attack_cat=fit_df['attack_cat'].values).to_parquet(OUT_DIR / 'fit.parquet')
X_val.assign(label=y_val, attack_cat=val_df['attack_cat'].values).to_parquet(OUT_DIR / 'val.parquet')
X_test.assign(label=y_test, attack_cat=test_raw['attack_cat'].values).to_parquet(OUT_DIR / 'test.parquet')
print(sorted(p.name for p in OUT_DIR.iterdir()))